In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
import os
import sys
sys.path.append("../")

from featurizer.pdb_feature import make_feature, NUCLEOTIDES
from utils.rigid_utils import calc_angl_rot_tsl, merge_rot_tsl, rot_vec_mul, calc_rot_tsl_batch

In [ ]:
def build_template_bank(csv_path, pdb_dir, save_path="../resources/templates.pt"):
    print(f"🚀 [修正版] 开始构建全原子软模板库...")
    
    if not os.path.exists(csv_path):
        print(f"❌ 错误: 找不到 CSV 文件 {csv_path}")
        return

    df = pd.read_csv(csv_path)
    bank = {nt: [] for nt in NUCLEOTIDES}
    
    pbar = tqdm(total=len(df))
    
    for _, row in df.iterrows():
        pdb_id = row['pdb_id']
        pdb_path = os.path.join(pdb_dir, f"{pdb_id}.pdb")
        
        if not os.path.exists(pdb_path): continue
        
        try:
            feat = make_feature(pdb_path)
        except Exception:
            continue
            
        try:
            coords_coarse = torch.from_numpy(feat['coords_coarse']).float()
            coords_gt = torch.from_numpy(feat['coords_gt']).float()
            mask_gt = torch.from_numpy(feat['mask_gt']).bool()
            angles = torch.from_numpy(feat['angles_gt']).float() 
            mask_angles = torch.from_numpy(feat['mask_angles']).bool()
            seq = feat['seq']
        except KeyError:
            continue

        L = len(seq)
        if L == 0: continue

        # --- Frame 计算 ---
        p1 = coords_coarse[:, 0] # C4'
        p2 = coords_coarse[:, 1] # C1'
        p3 = coords_coarse[:, 2] # N
        rot_bb, trans_bb = calc_rot_tsl_batch(p1, p2, p3)
        
        # --- Torsion 计算 ---
        rot_tor, trans_tor = calc_angl_rot_tsl(angles)
        
        # --- 合并变换 ---
        rot_glob, trans_glob = merge_rot_tsl(rot_bb, trans_bb, rot_tor, trans_tor)
        
        # --- 逆变换提取模板 ---
        has_valid = False
        for i in range(L):
            nt = seq[i]
            if nt not in NUCLEOTIDES: continue
            if not mask_angles[i]: continue
            if mask_gt[i].sum() < 3: continue
            
            curr_global = coords_gt[i]
            
            # 逆变换: Local = R^T * (Global - T)
            centered = curr_global - trans_glob[i]
            rot_inv = rot_glob[i].transpose(0, 1)
            curr_local = rot_vec_mul(rot_inv, centered)
            
            bank[nt].append({
                'pos': curr_local,
                'mask': mask_gt[i]
            })
            pbar.update(1)

    pbar.close()
    
    # --- 平均化 ---
    print("\n📊 计算平均坐标...")
    final_templates = torch.zeros((4, 14, 3)) 
    final_masks = torch.zeros((4, 14)).bool()
    nt_map = {nt: i for i, nt in enumerate(NUCLEOTIDES)}
    
    for nt, samples in bank.items():
        if not samples: continue
        idx = nt_map[nt]
        
        all_pos = torch.stack([s['pos'] for s in samples])
        all_mask = torch.stack([s['mask'] for s in samples])
        
        # Majority Vote Mask
        avg_presence = all_mask.float().mean(dim=0)
        final_masks[idx] = (avg_presence > 0.5)
        
        for atom_i in range(14):
            if final_masks[idx, atom_i]:
                valid = all_mask[:, atom_i]
                valid_coords = all_pos[:, atom_i, :][valid]
                if len(valid_coords) > 0:
                    final_templates[idx, atom_i, :] = valid_coords.mean(dim=0)

    torch.save({"pos": final_templates, "mask": final_masks}, save_path)
    print(f"✅ 修正后的模板库已保存: {save_path}")

In [ ]:
CSV_PATH = '../resources/gold_train.csv'
PDB_DIR = '../resources/pdbs'
build_template_bank(CSV_PATH, PDB_DIR)